In [ ]:
import re
import numpy as np
from pathlib import Path

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
# Regex to match Unicode superscript digits (⁰¹²³⁴⁵⁶⁷⁸⁹)
SUPERSCRIPT_UNICODE = r'[\u00b2\u00b3\u00b9\u2070\u2074\u2075\u2076\u2077\u2078\u2079]+'

# Primary split boundary: newline followed by NPPF paragraph number
# e.g. '\n237. ' triggers a split
PARA_BOUNDARY = r'\n(?=\d{1,3}\. )'

# Fallback splitter settings for paragraphs over 1000 chars
FALLBACK_CHUNK_SIZE = 1000
FALLBACK_OVERLAP    = 100

# File paths — adjust if your layout differs
PDF_PATH        = Path('../data/london_plan.pdf')
#EMBEDDINGS_PATH = Path('../data/nppf_embeddings.npy')
#CHROMA_PATH     = '../data/chroma_db'

print('Constants set')

In [ ]:
import re

# ---------------------------------------------------------------------------
# GG identifier restoration
# ---------------------------------------------------------------------------
GG_LOOKUP = {
    "GG Building strong": "GG1 Building strong",
    "GG Making the best": "GG2 Making the best",
    "GG Creating a healthy": "GG3 Creating a healthy",
    "GG Delivering the homes": "GG4 Delivering the homes",
    "GG Growing a good": "GG5 Growing a good",
    "GG Increasing efficiency": "GG6 Increasing efficiency",
}

def restore_gg_identifiers(text: str) -> str:
    for bad, good in GG_LOOKUP.items():
        text = text.replace(bad, good)
    return text


# ---------------------------------------------------------------------------
# Inline superscript stripping
# ---------------------------------------------------------------------------
def strip_inline_superscripts(text: str) -> str:
    """
    Remove inline footnote superscript numbers from running prose.
    
    Targets: 1-2 digits after a word character that are NOT:
      - Part of a section number like 3.11.2 (digit before AND after a decimal)
      - Part of a policy code like SD1 followed by a space and uppercase title
    
    Key insight from raw text analysis:
      - "COBR32 meetings"  -> strip 32 (followed by space+lowercase)
      - "Register33. The"  -> strip 33 (followed by period+space, NOT period+digit)
      - "design35. This"   -> strip 35 (same)
      - "SD1 Building"     -> keep 1 (followed by space+uppercase = policy title)
      - "3.11.2"           -> keep all (period followed by digit = section number)
    
    The period protection should only fire when the period is followed by a digit
    (decimal in a section number), not when it's a sentence-ending full stop.
    """
    return re.sub(
        r'(?<=[a-zA-Z\"\'\)])[,.]?(\d{1,2})(?!\d)(?!\.\d)(?!\s+[A-Z])',
        '',
        text
    )


# ---------------------------------------------------------------------------
# Continuation fragment extraction
# ---------------------------------------------------------------------------
def extract_continuation_fragment(text: str) -> tuple[str, str]:
    """
    Extract a misplaced continuation fragment from the end of a page.

    Some London Plan pages have a prose text box that visually appears at the
    TOP of the page but is placed LAST in the PDF content stream. pypdf extracts
    it at the end of the page text, after the footnote block.

    We locate it by finding everything after the last URL in the footnote block,
    up to the page footer. We then clean the leading URL path artefact (e.g.
    "crowded-places" — the tail of a wrapped URL that pypdf extracted on a
    new line without the http:// prefix).

    Returns (fragment, remaining_text). fragment is "" if none detected.
    """
    footer_match = re.search(r'\d+\s+The London Plan\s+\d+', text)
    footer_start = footer_match.start() if footer_match else len(text)

    text_before_footer = text[:footer_start]
    url_matches = list(re.finditer(r'https?://\S+|www\.\S+', text_before_footer))

    if not url_matches:
        return "", text

    last_url_end = url_matches[-1].end()
    candidate = text[last_url_end:footer_start].strip()

    # Strip leading URL path-continuation artefact:
    # e.g. "crowded-places fire, flood..." — "crowded-places" is the tail of a
    # wrapped URL. A path segment is lowercase letters and hyphens with no spaces,
    # followed by a space or newline before real prose begins.
    candidate = re.sub(r'^[a-z][a-z0-9\-]+\s+', '', candidate)

    if (candidate
            and len(candidate) > 30
            and re.match(r'^[a-zA-Z]', candidate)):
        remaining = text[:last_url_end] + text[footer_start:]
        return candidate, remaining

    return "", text


# ---------------------------------------------------------------------------
# Footnote block stripping
# ---------------------------------------------------------------------------
def strip_footnote_block(text: str) -> str:
    """
    Strip the footnote block at the bottom of a London Plan page.
    
    From raw text inspection, footnote entries appear as:
        \\n32 COBR (often referred to...
        \\n33 For further details...
    
    Pattern: newline, then 1-2 digits, then a single space, then an uppercase
    letter. We anchor to the newline to avoid matching mid-line numbers.
    """
    match = re.search(r'\n(\d{1,2}) (?=[A-Z])', text)
    if match:
        return text[:match.start()].strip()
    return text
# ---------------------------------------------------------------------------
# Main page cleaner
# ---------------------------------------------------------------------------
def clean_london_plan_page(text: str) -> str:
    """
    Clean a single page of raw pypdf text from the London Plan 2021.

    Pipeline:
      1. Strip "To table of contents" navigation artefact
      2. Strip page footer
      3. Extract misplaced continuation fragment (before footnote strip,
         so we can locate it relative to the last URL)
      4. Strip inline superscripts from main body text
      5. Strip footnote block
      6. Strip inline superscripts from the fragment (it was extracted before
         step 4, so needs its own pass)
      7. Strip URL path-continuation artefacts from fragment start
      8. Prepend fragment
      9. Strip orphaned standalone section numbers
     10. Restore GG identifiers
     11. Normalise whitespace
    """

    # 1. "To table of contents"
    text = re.sub(
        r'[\u2191\u2192\u2193\u2190\u25b2\u25bc]?\s*T\s?o\s+table\s+of\s+contents\s*',
        '', text, flags=re.IGNORECASE
    )

    # 2. Page footer
    text = re.sub(r'\d+\s+The London Plan\s+\d+[^\n]*', '', text)

    # 3. Extract continuation fragment BEFORE footnote stripping
    fragment, text = extract_continuation_fragment(text)

    # 4. Strip inline superscripts from main body
    text = strip_inline_superscripts(text)

    # 5. Strip footnote block
    text = strip_footnote_block(text)

    # 6. Strip inline superscripts from fragment too
    if fragment:
        fragment = strip_inline_superscripts(fragment)

    # 7. Prepend fragment
    if fragment:
        text = fragment + '\n\n' + text

    # 8. Strip orphaned standalone section numbers
    text = re.sub(r'(?m)^(\d{1,3})\n(?=\d+\.\d+\.\d+)', '', text)

    # 9. Restore GG identifiers
    text = restore_gg_identifiers(text)

    # 10. Normalise whitespace
    text = re.sub(r'[^\S\n]+', ' ', text)
    text = re.sub(r' \n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [ ]:
reader = PdfReader(PDF_PATH)
print(f"Total pages: {len(reader.pages)}")

# Extract and clean each page; store as list of dicts with page metadata
lp_pages = []
for i, page in enumerate(reader.pages):
    raw = page.extract_text() or ""
    cleaned = clean_london_plan_page(raw)
    lp_pages.append({
        "page_number": i + 1,
        "text": cleaned,
        "char_count": len(cleaned),
    })

print(f"Pages extracted: {len(lp_pages)}")
print(f"Pages with <50 chars (likely image-only or blank): {sum(1 for p in lp_pages if p['char_count'] < 50)}")

In [ ]:
# Inspect the page from pdf parser
print(lp_pages[156]["text"])

Initial notes from pdf parsing of London Plan:

How the document lables/numbers the paragraphs: majority of paragraphs are numbered in the format X.X.X - starting from 0.0.1. Other format is text boxes that do not have paragraph numbers - eg GG1, GG2 etc boxes, Policy SD1, Policy SD2 etc boxes, Policy D1, Policy D2 etc boxes, Policy H1, Policy H2 etc boxes, Policy S1, Policy S2, Policy E1, Policy HC1, Policy G1, Policy SI 1, Policy T1, inc Policy T6.1 - T6.5, Policy DF1, Policy M1 but contained letter paras - A, B, C etc, with some sub-numbering 1), 2), 3), then some sub-letterging a), b), c) and then some further subbing i, ii, iii
Page 145 has Table3.2 which has rows labelled i, ii, iii, iv, v, vi
Issue 1 - "T o table of contents" sometimes with a  at the top or bottom of every page.
Issue 2 - "GG1 Building strong.." changed to "GG Building Strong" - good growth objectives, same for all instances of GG1, GG2 etc - which are quite common as the Good Growth objectives are often shortened to GG1, GG2 etc
Issue 3 - page 52 has a table of numbers but it has been parsed as plain text
Issue 4 - p78 superscript and footnotes not removed: "A non-statutory strategic
structure has now been put in its place to address them,6" - although far fewer footnotes in this document, so may not be such a big issue. p79 superscript and footnote successfully removed. Seems like superscripts sometimes follow a , or . and they are not removed, but those that just follow text are.
Issue 5 - document is quite image heavy. Not such an issue if the main thing to take from document is planning policy.
Issue 6 - p161 missed the paragraph at the start of the page "f
ire, flood and related hazards. Development should include measures to 
design out crime that – in proportion to the risk – deter terrorism, assist in the 
detection of terrorist activity and help mitigate its effects. These measures 
should be considered at the start of the design process to ensure they are 
inclusive and aesthetically integrated into the development and the wider 
area"

In [ ]:
# Join all cleaned page texts into a single string.
# We insert a double newline between pages to preserve paragraph breaks.
lp_full_text = "\n\n".join(p["text"] for p in lp_pages)
print(f"Total characters: {len(lp_full_text):,}")

In [ ]:
import re

# Primary split boundaries for the London Plan:
#   1. Paragraph numbers: X.X.X (e.g. 3.11.1, 0.0.1, 12.1.8)
#   2. Policy headings: "Policy SD1", "Policy T6.1", "Policy SI 1" etc.
#   3. Good Growth headings: "GG1", "GG2" etc.
#   4. Chapter headings: "Chapter 1", "Chapter 2" etc.
#   5. Annex headings: "Annex 1", "Annex 2" etc.
#
# The capturing group means re.split() keeps the delimiter in the output,
# so we can reattach it to the chunk it introduces.

LP_SPLIT_PATTERN = re.compile(
    r'\n('
    r'\d+\.\d+\.\d+'                     # paragraph number: X.X.X
    r'|Policy\s+[A-Z]+\s*\d+[\d.]*'     # policy code: Policy SD1, Policy T6.1
    r'|GG\d+'                            # Good Growth: GG1-GG6
    r'|Chapter\s+\d+'                    # Chapter heading: Chapter 1, Chapter 2
    r'|Annex\s+\d+'                      # Annex heading: Annex 1, Annex 2, Annex 3
    r')\s'
)

In [ ]:
def split_london_plan(full_text: str) -> list[dict]:
    """
    Split the London Plan into chunks at paragraph and policy boundaries.
    
    Each chunk is a dict with:
      - 'id': the paragraph number (e.g. '3.11.1') or policy code (e.g. 'Policy SD1')
      - 'text': the full text of that paragraph or policy box
      - 'source': 'london_plan'
    
    The split works by using re.split() with a capturing group. re.split()
    returns alternating elements: [text_before, delimiter, text_after, delimiter, ...]
    We reattach each delimiter to the text that follows it, since the delimiter
    IS the paragraph/policy identifier that starts that chunk.
    """
    parts = LP_SPLIT_PATTERN.split(full_text)
    
    chunks = []
    
    # parts[0] is everything before the first boundary — front matter / intro
    if parts[0].strip():
        chunks.append({
            'id': 'front_matter',
            'text': parts[0].strip(),
            'source': 'london_plan',
        })
    
    # Remaining parts alternate: [delimiter, text, delimiter, text, ...]
    # Starting from index 1, step through in pairs
    for i in range(1, len(parts) - 1, 2):
        chunk_id = parts[i].strip()
        chunk_text = parts[i + 1].strip() if i + 1 < len(parts) else ""
        
        # Reattach the identifier to the chunk text
        full_chunk = f"{chunk_id} {chunk_text}"
        
        chunks.append({
            'id': chunk_id,
            'text': full_chunk,
            'source': 'london_plan',
        })
    
    return chunks


lp_chunks_raw = split_london_plan(lp_full_text)
print(f"Raw chunks after primary split: {len(lp_chunks_raw)}")

In [ ]:
import numpy as np

char_counts = [len(c['text']) for c in lp_chunks_raw]
print(f"Chunks: {len(char_counts)}")
print(f"Mean chars: {np.mean(char_counts):.0f}")
print(f"Median chars: {np.median(char_counts):.0f}")
print(f"Max chars: {np.max(char_counts)} (id: {lp_chunks_raw[np.argmax(char_counts)]['id']})")
print(f"Min chars: {np.min(char_counts)}")
print(f"Chunks > 1500 chars: {sum(1 for c in char_counts if c > 1500)}")
print(f"Chunks < 50 chars: {sum(1 for c in char_counts if c < 50)}")

In [ ]:
# Show the 10 longest chunks — these are likely policy boxes with many sub-clauses
long_chunks = sorted(lp_chunks_raw, key=lambda c: len(c['text']), reverse=True)[:10]
for c in long_chunks:
    print(f"  {c['id']:25s}  {len(c['text']):,} chars  |  first 80: {c['text'][:80]}...")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "],
)

MAX_CHUNK_SIZE = 1500
MIN_CHUNK_SIZE = 50

def split_with_fallback(chunks_raw: list[dict]) -> list[dict]:
    """
    Apply recursive splitting to any chunk exceeding MAX_CHUNK_SIZE.
    Drop chunks below MIN_CHUNK_SIZE (likely header artefacts or stray text).
    Sub-chunks inherit the parent's id with a suffix: e.g. 'Policy SD1_part1'
    """
    final_chunks = []
    
    for chunk in chunks_raw:
        text = chunk['text'].strip()
        
        # Skip tiny artefacts
        if len(text) < MIN_CHUNK_SIZE:
            continue
        
        if len(text) <= MAX_CHUNK_SIZE:
            final_chunks.append({
                'id': chunk['id'],
                'text': text,
                'source': chunk['source'],
            })
        else:
            # Recursive split
            sub_texts = recursive_splitter.split_text(text)
            for j, sub in enumerate(sub_texts):
                if len(sub.strip()) < MIN_CHUNK_SIZE:
                    continue  # skip tiny sub-chunks too
                final_chunks.append({
                    'id': f"{chunk['id']}_part{j+1}" if len(sub_texts) > 1 else chunk['id'],
                    'text': sub.strip(),
                    'source': chunk['source'],
                })
    
    return final_chunks


lp_chunks = split_with_fallback(lp_chunks_raw)
print(f"Final chunks after recursive splitting: {len(lp_chunks)}")

In [ ]:
final_counts = [len(c['text']) for c in lp_chunks]
print(f"Final chunk count: {len(final_counts)}")
print(f"Mean chars: {np.mean(final_counts):.0f}")
print(f"Median chars: {np.median(final_counts):.0f}")
print(f"Max chars: {np.max(final_counts)}")
print(f"Min chars: {np.min(final_counts)}")

# Breakdown by type
para_chunks = [c for c in lp_chunks if re.match(r'\d+\.\d+\.\d+', c['id'])]
policy_chunks = [c for c in lp_chunks if c['id'].startswith('Policy')]
gg_chunks = [c for c in lp_chunks if c['id'].startswith('GG')]
other_chunks = [c for c in lp_chunks if c not in para_chunks + policy_chunks + gg_chunks]

print(f"\nBy type:")
print(f"  Paragraph chunks:  {len(para_chunks)}")
print(f"  Policy chunks:     {len(policy_chunks)}")
print(f"  GG chunks:         {len(gg_chunks)}")
print(f"  Other (front matter etc): {len(other_chunks)}")

In [ ]:
# Show a few chunks from different categories
print("=== PARAGRAPH CHUNK ===")
sample_para = next(c for c in lp_chunks if c['id'] == '3.11.1')
print(f"ID: {sample_para['id']}")
print(f"Text: {sample_para['text'][:300]}")
print()

print("=== POLICY CHUNK ===")
sample_policy = next(c for c in lp_chunks if c['id'].startswith('Policy H1'))
print(f"ID: {sample_policy['id']}")
print(f"Text: {sample_policy['text'][:300]}")
print()

print("=== GG CHUNK (may have _part suffix) ===")
sample_gg = next(c for c in lp_chunks if c['id'].startswith('GG1'))
print(f"ID: {sample_gg['id']}")
print(f"Text: {sample_gg['text'][:300]}")
print()

print("=== ANNEX CHUNK ===")
sample_annex = next((c for c in lp_chunks if c['id'].startswith('Annex')), None)
if sample_annex:
    print(f"ID: {sample_annex['id']}")
    print(f"Text: {sample_annex['text'][:300]}")
else:
    print("No Annex chunks found")

In [49]:
# Regex to match Unicode superscript digits (⁰¹²³⁴⁵⁶⁷⁸⁹)
SUPERSCRIPT_UNICODE = r'[\u00b2\u00b3\u00b9\u2070\u2074\u2075\u2076\u2077\u2078\u2079]+'

# Matches digits glued directly after a letter — the signature of a
# superscript footnote reference merged into the word by pypdf.
# e.g. 'need85 ' or 'stage88.' → matched
# '12 March' or '2024,'      → NOT matched (space before digit)
INLINE_FOOTNOTE = r'(?<=[a-zA-Z])\d{1,3}(?=[\s\.,;:\)\-]|$)'

# Primary split boundary: newline followed by NPPF paragraph number
# e.g. '\n237. ' triggers a split
PARA_BOUNDARY = r'\n(?=\d{1,3}\. )'

# Fallback splitter settings for paragraphs over 1000 chars
FALLBACK_CHUNK_SIZE = 1000
FALLBACK_OVERLAP    = 100

# File paths — adjust if your layout differs
PDF_PATH        = Path('../data/nppf.pdf')

print('Constants set')

def clean_page_text(text: str) -> str:
    """
    Clean a single page of pypdf-extracted NPPF text.

    Step 1: Remove Unicode superscript digits.
    Step 2: Remove ASCII inline footnote reference numbers glued to words
            (e.g. 'need85' -> 'need'). Track which numbers are removed —
            these are the genuine footnote reference numbers for this page.
    Step 3: Remove the footnote block at the bottom of the page. A line is
            treated as the start of the footnote block only if its leading
            number was one of the tracked inline refs. This prevents false
            positives from body text like '12 December 2024'.
    Step 4: Normalise whitespace.
    """
    # Step 1: Unicode superscripts
    text = re.sub(SUPERSCRIPT_UNICODE, '', text)

    # Step 2: ASCII inline footnotes — remove and track reference numbers
    stripped_refs = set()

    def _remove_and_track(m):
        n = int(m.group())
        if n <= 120:
            stripped_refs.add(n)
            return ''
        return m.group()

    text = re.sub(INLINE_FOOTNOTE, _remove_and_track, text)

    # Step 3: Remove footnote block
    lines = text.split('\n')
    footnote_start = None

    if stripped_refs:
        for i, line in enumerate(lines):
            m = re.match(r'^(\d{1,3})\s+[A-Z]', line.strip())
            if m and int(m.group(1)) in stripped_refs:
                footnote_start = i
                break

    if footnote_start is not None:
        lines = lines[:footnote_start]

    text = '\n'.join(lines)

    # Step 4: Normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = '\n'.join(line.rstrip() for line in text.split('\n'))

    return text.strip()


print('clean_page_text() defined')

def hybrid_chunk(full_text: str) -> list:
    """
    Split NPPF text using a two-level strategy:

    Primary:  split on NPPF paragraph boundaries (\n\d{1,3}. )
              Each numbered paragraph becomes its own chunk.
    Fallback: any chunk over FALLBACK_CHUNK_SIZE chars is further
              split with RecursiveCharacterTextSplitter.

    Returns a list of dicts:
        text      — the chunk string
        chunk_id  — sequential integer index
        para_num  — NPPF paragraph number if detectable, else None
    """
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=FALLBACK_CHUNK_SIZE,
        chunk_overlap=FALLBACK_OVERLAP,
        separators=['\n\n', '\n', ' '],
    )

    raw_splits = re.split(PARA_BOUNDARY, full_text)

    chunks = []
    chunk_id = 0

    for split in raw_splits:
        split = split.strip()
        if not split:
            continue

        # Extract leading paragraph number for metadata if present
        para_match = re.match(r'^(\d{1,3})\. ', split)
        para_num = int(para_match.group(1)) if para_match else None

        if len(split) <= FALLBACK_CHUNK_SIZE:
            chunks.append({
                'text':     split,
                'chunk_id': chunk_id,
                'para_num': para_num,
            })
            chunk_id += 1
        else:
            # Paragraph too large — apply recursive fallback
            sub_chunks = fallback_splitter.split_text(split)
            for j, sub in enumerate(sub_chunks):
                chunks.append({
                    'text':     sub,
                    'chunk_id': chunk_id,
                    'para_num': para_num if j == 0 else None,
                })
                chunk_id += 1

    return chunks


print('hybrid_chunk() defined')

print(f'Loading PDF from {PDF_PATH} ...')
reader = PdfReader(PDF_PATH)

cleaned_pages = []
for i, page in enumerate(reader.pages):
    raw = page.extract_text()
    if raw:
        cleaned_pages.append(clean_page_text(raw))

full_text = '\n\n'.join(cleaned_pages)
print(f'Pages loaded: {len(cleaned_pages)}')

print('Chunking ...')
nppf_chunks = hybrid_chunk(full_text)

texts  = [c['text']     for c in nppf_chunks]
ids    = [c['chunk_id'] for c in nppf_chunks]
paras  = [c['para_num'] for c in nppf_chunks]

sizes = [len(t) for t in texts]
print(f'Total chunks : {len(nppf_chunks)}')
print(f'Char lengths  — mean: {np.mean(sizes):.0f}, '
      f'median: {np.median(sizes):.0f}, '
      f'max: {max(sizes)}, min: {min(sizes)}')

# Spot check — print paragraph 237 to verify cleaning is correct
target = next((c for c in nppf_chunks if c['para_num'] == 237), None)
if target:
    print(f'\n--- Paragraph 237 (spot check) ---')
    print(target['text'])
else:
    print('Paragraph 237 not found as a standalone chunk (may have been merged)')

Constants set
clean_page_text() defined
hybrid_chunk() defined
Loading PDF from ..\data\nppf.pdf ...
Pages loaded: 82
Chunking ...
Total chunks : 356
Char lengths  — mean: 524, median: 517, max: 1000, min: 2

--- Paragraph 237 (spot check) ---
237. Those local plans that reach Regulation 19 (pre-submission stage) on or before 12
March 2025 and whose draft housing requirement meets less than 80% of local
housing need should proceed to examination within a maximum of 18 months from
12 December 2024, or 24 months of that date if the plan has to return to the
Regulation 18 stage.


In [50]:
# Check we have both sets of chunks
print(f"NPPF chunks:       {len(nppf_chunks)}")  # from your earlier notebook
print(f"London Plan chunks: {len(lp_chunks)}")
print(f"Combined:          {len(nppf_chunks) + len(lp_chunks)}")

NPPF chunks:       356
London Plan chunks: 1504
Combined:          1860


In [52]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="chroma_db")

COLLECTION_NAME = "planning_policy"

# get_or_create avoids the NotFoundError entirely — if the collection exists
# it returns it, if not it creates it fresh. We then check the count and
# delete/recreate if it already has documents (so we always start clean).
existing = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if existing.count() > 0:
    print(f"Collection '{COLLECTION_NAME}' exists with {existing.count()} docs — deleting and recreating")
    client.delete_collection(COLLECTION_NAME)
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )
else:
    print(f"Collection '{COLLECTION_NAME}' is empty or new — ready to ingest")
    collection = existing

print(f"Collection ready: '{COLLECTION_NAME}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection 'planning_policy' is empty or new — ready to ingest
Collection ready: 'planning_policy'


In [53]:
import numpy as np

# Normalise both chunk formats into a consistent structure.
# NPPF chunks have keys: text, chunk_id, para_num  (no 'source')
# LP chunks have keys:   text, id, source

def normalise_chunk(chunk: dict, default_source: str) -> dict:
    return {
        'text':   chunk['text'],
        'id':     str(chunk.get('id', chunk.get('chunk_id', '?'))),
        'source': chunk.get('source', default_source),
    }

nppf_normalised = [normalise_chunk(c, 'nppf') for c in nppf_chunks]
lp_normalised   = [normalise_chunk(c, 'london_plan') for c in lp_chunks]

all_chunks = nppf_normalised + lp_normalised
print(f"NPPF chunks:        {len(nppf_normalised)}")
print(f"London Plan chunks: {len(lp_normalised)}")
print(f"Total to ingest:    {len(all_chunks)}")

BATCH_SIZE = 100

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]

    texts     = [c['text']   for c in batch]
    # IDs must be unique strings across the whole collection
    ids       = [f"{c['source']}__{c['id']}__{i + j}" for j, c in enumerate(batch)]
    metadatas = [{'source': c['source'], 'chunk_id': c['id']} for c in batch]

    embeddings = model.encode(texts).tolist()

    collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas,
    )

    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Ingested {min(i + BATCH_SIZE, len(all_chunks)):,} / {len(all_chunks):,}")

print(f"\nDone. Total documents in collection: {collection.count()}")

NPPF chunks:        356
London Plan chunks: 1504
Total to ingest:    1860
  Ingested 100 / 1,860
  Ingested 600 / 1,860
  Ingested 1,100 / 1,860
  Ingested 1,600 / 1,860

Done. Total documents in collection: 1860


In [54]:
def query_planning_policy(question: str, n_results: int = 5):
    """Query the combined planning policy collection."""
    embedding = model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[embedding],
        n_results=n_results,
    )
    
    print(f"Q: {question}\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    )):
        print(f"  [{i+1}] {meta['source']} | {meta['chunk_id']} | dist: {dist:.3f}")
        print(f"      {doc[:150]}...")
        print()

# Test queries that should pull from different documents
query_planning_policy("What is the affordable housing threshold in London?")

Q: What is the affordable housing threshold in London?

  [1] london_plan | 4.6.8 | dist: 0.281
      4.6.8 Currently all intermediate rented products such as London Living Rent and
Discounted Market Rent should be affordable to households on incomes o...

  [2] london_plan | 4.4.1 | dist: 0.309
      4.4.1 Delivering more genuinely affordable housing is a key strategic issue for
London. Meeting the need for circa 43,500 affordable homes per year, a...

  [3] london_plan | 4.6.9 | dist: 0.313
      4.6.9 For dwellings to be considered affordable, annual housing costs, including
mortgage (assuming reasonable interest rates and deposit requirements...

  [4] london_plan | 1.4.2 | dist: 0.321
      1.4.2 The state of London’s housing market has implications for the makeup and
diversity of the city. Affordable housing is central to allowing London...

  [5] london_plan | 4.5.4 | dist: 0.329
      4.5.4 The thresholds set out in this policy have been informed by viability testing. This
appr

In [55]:
query_planning_policy("What does national policy say about the Green Belt?")

Q: What does national policy say about the Green Belt?

  [1] nppf | 197 | dist: 0.243
      142. The Government attaches great importance to Green Belts. The fundamental aim of
Green Belt policy is to prevent urban sprawl by keeping land perm...

  [2] nppf | 151 | dist: 0.276
      108. Policies and decisions for managing development within a Local Green Space should
be consistent with national policy for Green Belts set out in c...

  [3] nppf | 199 | dist: 0.310
      144. The general extent of Green Belts across the country is already established. New
Green Belts should only be established in exceptional circumstan...

  [4] nppf | 206 | dist: 0.334
      151. Once Green Belts have been defined, local planning authorities should plan
positively to enhance their beneficial use, such as looking for opport...

  [5] london_plan | Policy G2 | dist: 0.338
      Policy G2 London’s Green Belt
A The Green Belt should be protected from inappropriate development:
1) development proposals th

In [56]:
query_planning_policy("What are the parking standards for new residential development in London?")

Q: What are the parking standards for new residential development in London?

  [1] london_plan | Policy T6.1_part4 | dist: 0.241
      

Location Number of beds Maximum parking provi-
sion*
Outer London PTAL 4 1 – 2 Up to 0.5 - 0.75 spaces per
dwelling+
Outer London PTAL 4 3+ Up to 0...

  [2] nppf | 159 | dist: 0.262
      113. Maximum parking standards for residential and non-residential development should
only be set where there is a clear and compelling justification ...

  [3] nppf | 158 | dist: 0.264
      112. If setting local parking standards for residential and non-residential development,
policies should take into account:

a) the accessibility of t...

  [4] london_plan | Annex 3_part6 | dist: 0.281
      from London (000’s tonnes) 373
T able 10.1 - Indicative list of transport schemes 407
T able 10.2 - Minimum cycle parking standards 416
T able 10.3 - ...

  [5] london_plan | 10.6.1 | dist: 0.281
      10.6.1 T o manage London’s road network and ensure that people and bu